In [ ]:
import java.time.LocalDateTime
import java.time.temporal.ChronoUnit

fun LongArray.shift(amount: Int = 1) {
    val n = amount.coerceIn(0, size - 1)
    if (n == 0) return

    val retained = sliceArray(0 until size - n)

    for (i in n until size) {
        this[i] = retained[i - n]
    }

    for (i in 0 until n) {
        this[i] = 0
    }
}

class ItemVolumeHistory(val bins: LongArray = LongArray(TOTAL_BINS.toInt())) {

    fun startNextBin() {
        bins.shift(1)
    }

    fun add(amount: Long) {
        bins[0] += amount
    }

    fun getFirstNDays(days: Int): List<List<Long>> {
        if (days < 1) return emptyList()

        val numBinsToday = binsToday()
        val clampedDays = days.coerceIn(1, TOTAL_DAYS)
        val numBins = (clampedDays * BINS_PER_DAY).toInt()

        val today = bins.slice(0 until numBinsToday).reversed()
        if (clampedDays == 1) return listOf(today)

        val days: MutableList<List<Long>> = bins.slice(numBinsToday until numBins)
            .reversed()
            .chunked(BINS_PER_DAY.toInt())
            .take(clampedDays - 1)
            .toMutableList()
        days += today
        return days
    }

    private fun binsToday(): Int {
        val midnight = LocalDateTime.now()
            .withHour(0)
            .withMinute(0)
            .withSecond(0)

        val now = LocalDateTime.now()
        val minsSinceMidnight = ChronoUnit.MINUTES.between(midnight, now)
        return ceil(minsSinceMidnight / RESOLUTION.toDouble()).toInt()
    }
}

val RESOLUTION = 15L
val BINS_PER_DAY = 24 * (60 / RESOLUTION)
val TOTAL_DAYS = 90
val TOTAL_BINS = TOTAL_DAYS * BINS_PER_DAY

In [ ]:
import me.danny.shop.tracking.Graph
import java.util.concurrent.TimeUnit
import net.md_5.bungee.api.ChatColor

fun printGraph(points: List<Long>) {
    val graph = Graph.create(points = points, resolution = 1, timescale = TimeUnit.DAYS)
    graph.forEach {
        println(ChatColor.stripColor(it))
    }
}

In [ ]:
fun fakeData(floor: Long, ceiling: Long): ItemVolumeHistory {
    val bins = ItemVolumeHistory()
    (0 until TOTAL_BINS).forEach {
        bins.startNextBin()
        bins.add((floor..ceiling).random())
    }

    return bins
}

In [ ]:
val hist = fakeData(0, 100)
val week = hist.getFirstNDays(90).map(List<Long>::sum)
printGraph(week)

In [ ]:
import me.danny.shop.model.ID
import me.danny.shop.tracking.EcoAnalytics
import java.io.File
import java.io.FileInputStream
import java.io.ObjectInputStream
import java.io.Serializable
import java.time.Instant
import java.util.zip.GZIPInputStream

fun readHist(): EcoAnalytics.Data? {
    val file = File("/home/daniel/Downloads/newpaper/plugins/DannyShop/item-history.bin")
    try {
        val read = ObjectInputStream(GZIPInputStream(FileInputStream(file))).use(ObjectInputStream::readObject)
        if (read is EcoAnalytics.Data) {
//            println(read.lastTick)
//            println(read.history[ID("19ee8480548138")]!!.bins.toList())
            return read
        }
    } catch (ex: Exception) {
        println("Volume analytics: Error while loading item history from ${file.absolutePath}:")
        ex.printStackTrace()
    }

    return null
}

val data = readHist() ?: throw Exception("wat")
val wool = ID("19ee8480548138")
val today = data.history[wool]!!.getFirstNDays(2)
today.map(List<*>::size).forEach(::println)
println(today)

In [ ]:
val hist = ItemVolumeHistory()
hist.getFirstNDays(1).size